# Wakil AI — Phase 4: The Application

Your models become an app that anyone can use: **draw a digit**, **generate text**, and **see the models**.

### How the project is organized
```
app.py               ← the interface only (Gradio: buttons, canvas, sliders)
  │ calls
wakil_ai/service.py  ← the application logic: load models, prepare the drawing, predict, generate
  │ uses
wakil_ai/*.py        ← YOUR models: WakilNet, CharMLP, TinyGPT (NumPy, from scratch)
```
The interface never touches the math. To build a different interface later (web site, mobile app, Telegram/WhatsApp bot), you only replace `app.py`; the models don't change.

**Gradio** is only used for the interface (it draws the buttons in the browser). **All the intelligence is your NumPy code.**

**Runtime → Run all.** About 1 minute, plus ~4–8 minutes if you also train the Transformer (Cell 3).

## Cell 1 — Get the project and install Gradio

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yassine1158/WAKIL-AI.git"
if not os.path.exists("wakil_ai/service.py"):
    if not os.path.exists("WAKIL-AI"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("WAKIL-AI")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

!pip install -q -r requirements-app.txt
import gradio as gr
print("Gradio", gr.__version__)

## Cell 2 — Train the digit reader and the small language model (~1 minute)

These are the same commands you ran on your PC.

In [ ]:
!python train.py
!python evaluate.py
!python train_lm.py

## Cell 3 — (Optional) Train the Transformer (~4–8 minutes)

Set `TRAIN_TRANSFORMER = False` to skip it: the app then uses the MLP language model.

In [ ]:
TRAIN_TRANSFORMER = True

if TRAIN_TRANSFORMER:
    !python train_lm.py --arch transformer

## Cell 4 — Test the logic without any interface

This is the advantage of separating logic and interface: the service can be tested with plain Python (the project's `tests/test_service.py` does exactly that).

**Expected output:** a description of the loaded models, then a short text written by Wakil AI.

In [ ]:
from IPython.display import Markdown, display
from wakil_ai.service import WakilService

service = WakilService()
display(Markdown(service.info()))
text, message = service.write("ROMEO:\n", length=200, temperature=0.8, seed=1)
print(message)
print(text)

## Cell 5 — Launch the app 🚀

In Colab, Gradio shows the app **inside the notebook** and also prints a **public link** (`https://….gradio.live`). Open it on your phone or send it to a friend.

- The link works **only while this notebook is running** (and at most 1 week).
- ⚠️ Anyone who has the link can use your app while it runs. Don't share it publicly if you don't want to.
- To stop the app: **Runtime → Interrupt execution**, or close the notebook.

**How to use it:**
1. **✍️ Read a digit:** click the **🖌️ Brush** button, draw a big digit, then click **Read**.
2. **💬 Write text:** type a beginning, choose the temperature, then click **Write**.
3. **ℹ️ Models:** the size and the scores of your models.

In [ ]:
import importlib
import app as wakil_app

importlib.reload(wakil_app)            # reload so the app sees the models trained above
in_colab = "google.colab" in sys.modules
demo = wakil_app.build_app()
demo.launch(share=in_colab, prevent_thread_lock=True)

## Run it on your PC

In PowerShell, inside the project folder:
```powershell
python -m pip install -r requirements-app.txt
python app.py
```
Then open **http://127.0.0.1:7860** in your browser. Stop it with **Ctrl + C** in PowerShell.

## Phase 4 checklist
- [ ] The app opens (in Colab, on your PC, or through the public link)
- [ ] You drew 5 different digits: how many did Wakil AI read correctly? Look at "What the model sees" for the mistakes
- [ ] You generated text with temperature 0.3 and 1.5 and saw the difference
- [ ] The Models tab shows your accuracy and loss values
- [ ] You can explain why `app.py` and `wakil_ai/service.py` are separate files

## Ideas to go further
1. **Your own handwriting dataset:** collect your drawings and their labels from the app, and fine-tune the digit reader on them. This is the honest way to fix distribution shift.
2. **French or Arabic:** train `train_lm.py --text my_book.txt` on a public-domain book and add the model to the app.
3. **A convolutional network (CNN)** from scratch for the digits, to go above 99%.
4. **A word-level or BPE tokenizer**, the kind real LLMs use.